In [1]:


import pandas as pd
import numpy as np
import gc

from scipy.sparse import coo_matrix

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')




In [2]:
aisles = pd.read_csv(
    '/kaggle/input/instacart-market-basket-analysis/aisles.csv'
)

departments = pd.read_csv(
    '/kaggle/input/instacart-market-basket-analysis/departments.csv'
)

products = pd.read_csv(
    '/kaggle/input/instacart-market-basket-analysis/products.csv',
    usecols=['product_id','product_name','aisle_id','department_id']
)

orders = pd.read_csv(
    '/kaggle/input/instacart-market-basket-analysis/orders.csv',
    usecols=['order_id','user_id']
)

order_products_prior = pd.read_csv(
    '/kaggle/input/instacart-market-basket-analysis/order_products__prior.csv',
    usecols=['order_id','product_id']
)


In [3]:
N_USERS = 1800   # 

selected_users = orders['user_id'].drop_duplicates().sample(
    N_USERS,
    random_state=42
)

orders_small = orders[orders['user_id'].isin(selected_users)]

data = order_products_prior.merge(
    orders_small,
    on='order_id'
)

print("Final interaction shape:", data.shape)

Final interaction shape: (268897, 3)


### Merge with product features (Content-Based)

In [4]:
data = data.merge(
    products,
    on='product_id'
)

print("After adding product info:", data.shape)


After adding product info: (268897, 6)


In [5]:
data


,order_id,product_id,user_id,product_name,aisle_id,department_id
0,28,35108,98256,Salted Butter,36,16
1,28,40593,98256,Cream Cheese,108,16
2,28,17461,98256,Air Chilled Organic Boneless Skinless Chicken ...,35,12
3,28,22825,98256,Organic D'Anjou Pears,24,4
4,28,25256,98256,Cultured Low Fat Buttermilk,84,16
...,...,...,...,...,...,...
268892,3420938,42445,38426,Organic Plain Whole Milk Yogurt,120,16
268893,3420938,43086,38426,Super Greens Salad,123,4
268894,3420938,46226,38426,Thick & Crispy Tortilla Chips,107,19
268895,3420938,18770,38426,Uncured Italian Dry Salami,96,20


## Collaborative Filtering (SVD)

#### create user-item matrix

In [6]:
user_item_matrix = data.pivot_table(
    index='user_id',
    columns='product_id',
    values='order_id',
    aggfunc='count',
    fill_value=0
)

# تحويلها إلى binary interaction
user_item_matrix[user_item_matrix > 0] = 1

#### applied SVD

In [7]:
from sklearn.decomposition import TruncatedSVD
import numpy as np
import pandas as pd

svd = TruncatedSVD(n_components=50, random_state=42)

latent_matrix = svd.fit_transform(user_item_matrix)

collab_pred = np.dot(latent_matrix, svd.components_)

collab_df = pd.DataFrame(
    collab_pred,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)


## Content-Based Filtering

In [8]:
# merge products info
product_features = products.merge(departments, on='department_id')
product_features = product_features.merge(aisles, on='aisle_id')

product_features['text'] = (
    product_features['department'] + " " +
    product_features['aisle']
)


#### TF-IDF + Cosine Similarity

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer()

content_matrix = tfidf.fit_transform(product_features['text'])

content_similarity = cosine_similarity(content_matrix)

content_df = pd.DataFrame(
    content_similarity,
    index=product_features['product_id'],
    columns=product_features['product_id']
)


## Association Rules (FP-Growth)

In [10]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

transactions = data.groupby('order_id')['product_id'].apply(list)

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_trans = pd.DataFrame(te_array, columns=te.columns_)

freq_items = fpgrowth(df_trans, min_support=0.01, use_colnames=True)

rules = association_rules(freq_items, metric="lift", min_threshold=1)


## Hybrid Recommendation Function

In [11]:
def hybrid_recommend(user_id, n=10):

    # Collaborative filtering scores for the user
    collab_scores = collab_df.loc[user_id]

    # Products already purchased by the user
    purchased = user_item_matrix.loc[user_id]
    purchased = purchased[purchased > 0].index

    # Content-based score:
    # Average similarity between candidate products
    # and the products already purchased
    content_scores = content_df[purchased].mean(axis=1)

    # Initialize association rule scores
    assoc_scores = pd.Series(0, index=collab_scores.index)

    # Compute association rule contribution
    for product in purchased:

        matched = rules[
            rules['antecedents'].apply(lambda x: product in x)
        ]

        for _, row in matched.iterrows():
            for consequent in row['consequents']:
                assoc_scores[consequent] += row['lift']

    # Normalize scores to avoid scale dominance
    collab_scores /= collab_scores.max()
    content_scores /= content_scores.max()
    assoc_scores /= assoc_scores.max()

    # Final hybrid score (weighted combination)
    final_score = (
        0.5 * collab_scores +
        0.3 * content_scores +
        0.2 * assoc_scores
    )

    # Remove already purchased products
    final_score = final_score.drop(purchased, errors='ignore')

    # Return Top-N recommendations
    return final_score.sort_values(ascending=False).head(n)


In [12]:
hybrid_recommend(user_id=38426, n=5)


product_id
21137    0.593796
22935    0.471930
34243    0.449302
41950    0.447676
23165    0.424983
dtype: float64

In [13]:
recommendations = hybrid_recommend(user_id=38426, n=5)

# Transform DataFrame
recommendations = recommendations.reset_index()
recommendations.columns = ['product_id', 'score']

# 
recommendations = recommendations.merge(
    products[['product_id','product_name']],
    on='product_id',
    how='left'
)

recommendations[['product_id','product_name','score']]


,product_id,product_name,score
0,21137,Organic Strawberries,0.593796
1,22935,Organic Yellow Onion,0.471930
2,34243,Organic Baby Broccoli,0.449302
3,41950,Organic Tomato Cluster,0.447676
4,23165,Organic Leek,0.424983
